<a href="https://colab.research.google.com/github/fmarryam70-ux/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fmarryam70-ux/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh / Content Opportunity Scoring.

Task type: Ranking / Scoring, built on top of a classification signal.

Decision this improves: which content pages an editor should refresh FIRST,
out of thousands, given limited editor time.

Who acts: content editors/SEO strategists — they pick pages from a ranked
priority list and refresh them (update content, add keywords, expand word
count, etc).

Cost of a wrong answer: a false positive wastes editor hours refreshing a
page that wasn't actually declining. A false negative means a real
decliner keeps losing traffic silently. Both costs matter, so ranking
(not just a binary flag) lets editors triage by confidence/impact.

Why not pure classification: "declining or not" alone doesn't tell an
editor WHICH declining page to fix first when there are hundreds.
Scoring/ranking combines decline likelihood with opportunity size
(search_volume, competition) into one priority order.

In [87]:
# Task type: Ranking / Scoring (built on a classification signal)
# Lane: Content Refresh Opportunity Scoring

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_declining_label — already present in the prepared feature
table (from docs/data-dictionary.md). Defined as
is_declining_label = 1 when trend_direction == "down", i.e. impressions
in the last 30 days dropped more than 20% vs the previous 30 days.
16,262 of 30,000 rows (54.2%) are positive.

Important caveat: this label is RULE-DEFINED (a threshold on trend_pct),
not a purely observed later-window outcome. Per the framing-ml-problems
guidance, a rule-defined label risks the model just re-learning the
threshold rule instead of the real world pattern. I'm aware of this and
treat the label as a reasonable starting proxy, not ground truth.

Leakage rule (critical): trend_direction and trend_pct are NEVER used as
features — they are the source of the label itself.

For the final priority score (Section 1's ranking output), I combine:
predicted decline probability (from the classifier) × opportunity weight
(search_volume, competition, cpc) → one refresh-priority score per page.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

For the underlying classifier: ROC-AUC, and precision/recall compared
against the 54.2% base rate (since classes are fairly balanced, ROC-AUC
is meaningful here).

For the final ranked output (what editors actually see): precision@K —
of the top K pages flagged as highest priority, how many are actually
worth refreshing. This matters more than raw classification accuracy,
since editors only work through a limited list each week.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [88]:
import os
os.chdir('/content')
!rm -rf flyrank-ml-internship
!git clone https://github.com/fmarryam70-ux/flyrank-ml-internship.git
%cd flyrank-ml-internship
!ls data/raw/

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 129 (delta 43), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.84 MiB | 7.52 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/flyrank-ml-internship
content_refresh_anonymized.csv


In [89]:
!ls data/raw/

content_refresh_anonymized.csv


In [90]:
!cat skills/README.md

# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write +

In [91]:
!cat skills/framing-ml-problems/SKILL.md


---
name: framing-ml-problems
description: Frames a data/ML problem before any modeling — the decision, the action, the cost of a wrong call, task type, target, and success metric. Use when picking a project direction, writing a research question, or mapping a problem onto an ML task type.
---

# Framing ML problems

A model is never the goal. A better DECISION is the goal. Frame first, model later.

## The four questions (answer all four, in writing)

1. **What decision does this improve?** Not "predict X" — decisions. "Which page should an
   editor fix first?" is a decision. "Predict decline" is not, yet.
2. **Who acts on the output, and what do they do?** Name the person and the action. If nobody
   would act differently, the work has no customer.
3. **What does a wrong answer cost?** Wasted editor hours? A missed decline? The cost of errors
   decides how careful the method must be, and which errors matter more.
4. **Why does data or ML help at all?** Sometimes a plain rule (an if

In [92]:
!cat skills/flyrank/flyrank-data/SKILL.md

---
name: flyrank-data
description: The FlyRank internship datasets — the 30k-row starter CSV and its gotchas, the ~79M-row warehouse release tables and grains, panel warnings, access, and iteration rules. Load for EVERY task that touches the data. (Project-specific: delete this folder when reusing the skill library elsewhere.)
---

# FlyRank internship data

Two datasets. The small one ships in this repo; the big one is hosted and gated.

## 1. Starter dataset (in this repo)

`data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns, one row per pseudonymized
content item, 32 clients, trailing-90-day metrics. Full column reference: `docs/data-dictionary.md`
(keep it open). The gotchas that cause 90% of mistakes:

- **Rate columns are ×100 percentages**: `ctr = 0.76` means 0.76%, not 76%. Applies to ctr,
  engagement_rate, scroll_rate, ai_traffic_pct, trend_pct.
- **`avg_position = 0` means "no data"**, not rank zero (1,205 rows).
- **`scroll_rate` and `ai_traffic_pct` can e

In [93]:
with open("skills/framing-ml-problems/SKILL.md") as f:
    print(f.read())

---
name: framing-ml-problems
description: Frames a data/ML problem before any modeling — the decision, the action, the cost of a wrong call, task type, target, and success metric. Use when picking a project direction, writing a research question, or mapping a problem onto an ML task type.
---

# Framing ML problems

A model is never the goal. A better DECISION is the goal. Frame first, model later.

## The four questions (answer all four, in writing)

1. **What decision does this improve?** Not "predict X" — decisions. "Which page should an
   editor fix first?" is a decision. "Predict decline" is not, yet.
2. **Who acts on the output, and what do they do?** Name the person and the action. If nobody
   would act differently, the work has no customer.
3. **What does a wrong answer cost?** Wasted editor hours? A missed decline? The cost of errors
   decides how careful the method must be, and which errors matter more.
4. **Why does data or ML help at all?** Sometimes a plain rule (an if

In [94]:
with open("docs/data-dictionary.md") as f:
    print(f.read())

# Data Dictionary — `content_refresh_anonymized.csv`

One row per content item (page): **30,000 rows × 44 columns**, covering **32 pseudonymized
clients**. All metrics are aggregated over a trailing 90-day window ending at export time.
Keep this file open while you work.

## Read this first — the three rules that prevent 90% of mistakes

1. **Rate columns are ×100 percentages.** `ctr = 0.76` means **0.76%**, not 76%. Applies to
   `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`.
2. **The label comes from `trend_direction`.** The pipeline defines
   `is_declining_label = (trend_direction == "down")`, so `trend_direction` and `trend_pct`
   must **never** be model features — that's the leakage notebook 02 demonstrates.
3. **IDs are for grouping only.** `content_id` / `client_id` are pseudonyms: use them for
   joins and grouped train/test splits, never as features.

## Identifiers

| Column | Type | Meaning | Notes |
|---|---|---|---|
| `content_id` | text | Pseudo

In [96]:
import pandas as pd

# Load the lane's slice of the starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content page (unit of analysis)
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


Unit of analysis: one row = one content page (from content_refresh_anonymized.csv,
30,000 rows, 44 columns, 32 clients). Key columns include content_type, word_count,
search_volume, competition, engagement_rate, ai_traffic_pct, and the target
is_declining_label. No client names or URLs are present — content_id and client_id
are pseudonymized identifiers used only for grouping/joins.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule already exists in this data (trend_pct threshold defines
the label itself) — and that's exactly the problem: a single-signal
threshold is blunt. It only reacts AFTER a 20%+ drop has already
happened, and it ignores everything else that predicts decline earlier:
content_age_days, freshness_tier, word_count, engagement_rate,
ai_traffic_pct, content_type (missingness itself is informative here).

ML can combine these many, tangled, interacting signals to predict
decline risk before it fully shows up in trend_pct, and can weigh
opportunity size (search_volume, cpc) alongside risk — something a
single if/else rule cannot do without becoming an unmanageable pile of
nested conditions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.